In [30]:
import os

java_home = os.getenv("JAVA_HOME")

if not java_home:
    raise RuntimeError("JAVA_HOME environment variable is not configured")

print(f"✓ JAVA_HOME={java_home}")

✓ JAVA_HOME=/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home


# Análise — NY Yellow Taxi 2025 (Jan–Mai)

Responde as duas questões do case utilizando os datasets gold gerados pelo pipeline.

In [31]:
from pyspark.sql import functions as F
from ny_rides.shared.spark import get_spark_session

spark = get_spark_session("Questions")
print("Spark session ready")

Spark session ready


## Questão 1 — Qual a média de valor total (total_amount) recebido em um mês?

Considerando todos os yellow táxis da frota.

In [32]:
df_monthly_avg = (
    spark.read.parquet("../data/gold/monthly_average_total_amount")
    .filter((F.col("pickup_year") == 2025) & (F.col("pickup_month") <= 5))
    .orderBy("pickup_year", "pickup_month")
)
df_monthly_avg.show()

+-----------+------------+----------------+
|pickup_year|pickup_month|avg_total_amount|
+-----------+------------+----------------+
|       2025|           1|           25.61|
|       2025|           2|           25.03|
|       2025|           3|           26.27|
|       2025|           4|           26.59|
|       2025|           5|           26.88|
+-----------+------------+----------------+



## Questão 2 — Qual a média de passageiros (passenger_count) por cada hora do dia?

Considerando todos os táxis da frota que pegaram táxi no mês de maio.

In [33]:
# Read hourly average passenger count from gold and keep only May/2025
df_hourly_avg_may = (
    spark.read.parquet("../data/gold/hourly_average_passenger_count")
    .filter((F.col("pickup_year") == 2025) & (F.col("pickup_month") == 5))
    .orderBy("pickup_hour")
    .withColumn("avg_passenger_count", F.round(F.col("avg_passenger_count"), 2))
)

df_hourly_avg_may.show(24)

{"ts": "2026-06-15 20:41:18.537", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_year` cannot be resolved. Did you mean one of the following? [`pickup_hour`, `pickup_month`, `avg_passenger_count`]. SQLSTATE: 42703", "context": {"file": "line 4 in cell [34]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o560.filter.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_year` cannot be resolved. Did you mean one of the following? [`pickup_hour`, `pickup_month`, `avg_passenger_count`]. SQLSTATE: 42703;\n'Filter 'and('`=`('pickup_year, 2025), (pickup_month#442 = 5))\n+- Relation [pickup_month#442,pickup_hour#443,avg_passenger_count#444] parquet\n\n\tat org.apache.spark.sql.err

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_year` cannot be resolved. Did you mean one of the following? [`pickup_hour`, `pickup_month`, `avg_passenger_count`]. SQLSTATE: 42703;
'Filter 'and('`=`('pickup_year, 2025), (pickup_month#442 = 5))
+- Relation [pickup_month#442,pickup_hour#443,avg_passenger_count#444] parquet
